In [1]:
from importlib.resources import files
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ks_2samp

In [2]:
csv_path = files("data").joinpath("echonext_metadata_100k.csv")
df = pd.read_csv(csv_path, index_col=0)

Descriptions of features and targets. I decided to focus on predicting those binary labels, so that's how my analysis will go.

In [3]:
DEMOGRAPHIC_FEATURES = [
    "patient_key", # de-identified patient ID
    "acquisition_year", # year the ecg was acquired 
    "location_setting", # clinical context of the ecg acquisition
    "race_ethnicity", # self-explanatory
    "most_recent_ecg" # whether the ecg is the most recent one for the patient
]
ECG_DERIVED_FEATURES = [
    "sex", # self-explanatory
    "ventricular_rate", # ventricular rate in beats per minute (how fast the ventricles are beating)
    "atrial_rate", # atrial rate in beats per minute (how fast atria is depolarizing)
    "pr_interval", # pr interval in milliseconds (start from atrial depolarization to the start of ventricular depolarization)
    "qrs_duration", # qrs duration in milliseconds (duration of ventricular depolarization)
    "qt_corrected", # corrected qt interval in milliseconds (total time of ventricular depolarization and repolarization in a rate-independent way)
    "age_at_ecg" # age of the patient at the time of ecg acquisition
]
TARGETS = [
    "lvef_lte_45_flag", # LVEF ≤ 45% 
    "lvwt_gte_13_flag", # LV wall thickness ≥ 1.3 cm
    "aortic_stenosis_moderate_or_greater_flag", # Moderate or severe aortic stenosis
    "aortic_regurgitation_moderate_or_greater_flag", # Moderate or severe aortic regurgitation
    "mitral_regurgitation_moderate_or_greater_flag", # Moderate or severe mitral regurgitation
    "tricuspid_regurgitation_moderate_or_greater_flag", # Moderate or severe tricuspid regurgitation
    "pulmonary_regurgitation_moderate_or_greater_flag", # Moderate or severe pulmonary regurgitation
    "rv_systolic_dysfunction_moderate_or_greater_flag", # Moderate or severe RV dysfunction
    "pericardial_effusion_moderate_large_flag", # Moderate or large pericardial effusion
    "pasp_gte_45_flag", # PASP ≥ 45 mmHg
    "tr_max_gte_32_flag", # TR velocity ≥ 3.2 m/s
    "shd_moderate_or_greater_flag" # Composite label indicating presence of moderate or greater structural heart disease
]

## Analysis of the splits provided by authors

In [4]:
def summarize_split(name, df_split, feature_cols=None, target_cols=None):
    out = {
        "split": name,
        "n_samples": len(df_split),
    }

    if feature_cols is None:
        excluded = {"split", "patient_key"}
        if target_cols is not None:
            excluded |= set(target_cols)
        feature_cols = [c for c in df_split.columns if c not in excluded]

    out["n_features"] = len(feature_cols)

    if feature_cols:
        missing_rates = df_split[feature_cols].isna().mean()
        out["mean_feature_missing_rate"] = float(missing_rates.mean())
        out["max_feature_missing_rate"] = float(missing_rates.max())
    else:
        out["mean_feature_missing_rate"] = np.nan
        out["max_feature_missing_rate"] = np.nan

    if target_cols is not None:
        for col in target_cols:
            if col in df_split.columns:
                out[f"{col}_prevalence"] = float(df_split[col].mean())
                out[f"{col}_missing_rate"] = float(df_split[col].isna().mean())

    return out

In [5]:
df_train = df[df["split"] == "train"]
df_no_split = df[df["split"] == "no_split"]
df_val = df[df["split"] == "val"]
df_test = df[df["split"] == "test"]

In [6]:
summary_train = summarize_split("train", df_train, ECG_DERIVED_FEATURES, TARGETS)
summary_no_split = summarize_split("no_split", df_no_split, ECG_DERIVED_FEATURES, TARGETS)
summary_val = summarize_split("val", df_val, ECG_DERIVED_FEATURES, TARGETS)
summary_test = summarize_split("test", df_test, ECG_DERIVED_FEATURES, TARGETS)

In [7]:
pd.DataFrame([summary_train, summary_no_split, summary_val, summary_test]).set_index("split").T

split,train,no_split,val,test
n_samples,72475.000000,17457.000000,4626.000000,5442.000000
n_features,7.000000,7.000000,7.000000,7.000000
mean_feature_missing_rate,0.015911,0.015115,0.015441,0.014805
max_feature_missing_rate,0.105126,0.100934,0.101383,0.095369
lvef_lte_45_flag_prevalence,0.234039,0.292261,0.187203,0.176773
lvef_lte_45_flag_missing_rate,0.000000,0.000000,0.000000,0.000000
lvwt_gte_13_flag_prevalence,0.243767,0.264364,0.189581,0.194965
lvwt_gte_13_flag_missing_rate,0.000000,0.000000,0.000000,0.000000
aortic_stenosis_moderate_or_greater_flag_prevalence,0.040276,0.034198,0.054475,0.052554
aortic_stenosis_moderate_or_greater_flag_missing_rate,0.000000,0.000000,0.000000,0.000000


Conclusions from comparing splits provided by the authors:

1. Split sizes are very unbalanced. Train is much larger compared to val and test, so uncertainty on rare labels will be higher in val/test.
2. Feature completness looks stable across splits.
3. All target labels are complete in every split. Additionally SHDs are more present in the train split then val or test. One exception from this rule is aortic stenosis.
4. No split has more prevelance of SHDs than other splits (different distribution - it can be additionally used to check robustness of the final model)
5. Val and test are very similar, which is good.

Now let's check patient overlap and if test and val only contains the latest ECG as authors mentioned.

In [8]:
p_train = set(df.loc[df["split"] == "train", "patient_key"].dropna().unique())
p_val = set(df.loc[df["split"] == "val", "patient_key"].dropna().unique())
p_test = set(df.loc[df["split"] == "test", "patient_key"].dropna().unique())
p_no_split = set(df.loc[df["split"] == "no_split", "patient_key"].dropna().unique())

overlaps = {
    "train_val": len(p_train & p_val),
    "train_test": len(p_train & p_test),
    "val_test": len(p_val & p_test),
    "train_no_split": len(p_train & p_no_split),
    "val_no_split": len(p_val & p_no_split),
    "test_no_split": len(p_test & p_no_split),
}

overlaps

{'train_val': 0,
 'train_test': 0,
 'val_test': 0,
 'train_no_split': 0,
 'val_no_split': 2119,
 'test_no_split': 2499}

In [9]:
# percentage of latest ECG per split
latest_pct = (
    df.groupby("split")["most_recent_ecg"]
    .agg(
        n="size",
        latest_count=lambda s: (s == 1).sum(),
        latest_pct=lambda s: (s == 1).mean() * 100
    )
    .sort_index()
)

latest_pct

,n,latest_count,latest_pct
split,,,
no_split,17457,0,0.000000
test,5442,5442,100.000000
train,72475,26218,36.175233
val,4626,4626,100.000000


There's no overlap between train, val and test - no patient leakeage.
No split is not independent from val and test. From the data analysis it shows that no_split has historical ECG from the patients present in test and val.

In [10]:
p_blocked = p_val | p_test
p_no_clean = p_no_split - p_blocked

print("no_split unique patients:", len(p_no_split))
print("no_split patients overlapping val/test:", len(p_no_split & p_blocked))
print("no_split patients NOT in val/test:", len(p_no_clean))

no_split unique patients: 4618
no_split patients overlapping val/test: 4618
no_split patients NOT in val/test: 0


So I can't use no split for training because of patient overlap. I can just use it as a supplementary benchmark.

Let's also check if features have similar distributions across splits.

In [11]:
categorical_features = [
    "location_setting",
    "race_ethnicity",
    "sex"
]
continuous_features = [
    feature for feature in ECG_DERIVED_FEATURES if feature not in categorical_features
]

In [12]:
def _cramers_v_from_counts(counts, chi2):
    n = counts.to_numpy().sum()
    r, k = counts.shape

    if n == 0:
        return np.nan

    phi2 = chi2 / n

    # bias correction
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    rcorr = r - ((r - 1) ** 2) / (n - 1)
    kcorr = k - ((k - 1) ** 2) / (n - 1)

    denom = min((kcorr - 1), (rcorr - 1))
    if denom <= 0:
        return np.nan

    return np.sqrt(phi2corr / denom)


def compare_categorical_distributions(
    df,
    categorical_cols,
    split_col="split",
    splits=("train", "val", "test", "no_split"),
):
    proportion_tables = {}
    stats_rows = []

    work = df[df[split_col].isin(splits)].copy()

    for col in categorical_cols:
        if col not in work.columns:
            continue

        counts = pd.crosstab(work[col].fillna("<MISSING>"), work[split_col])
        counts = counts.reindex(columns=list(splits), fill_value=0)

        props = counts.div(counts.sum(axis=0), axis=1)
        proportion_tables[col] = props

        try:
            chi2, p_value, dof, _ = chi2_contingency(counts)
            cramers_v = _cramers_v_from_counts(counts, chi2)
        except ValueError:
            chi2, p_value, dof, cramers_v = np.nan, np.nan, np.nan, np.nan

        stats_rows.append(
            {
                "feature": col,
                "n_levels": counts.shape[0],
                "chi2": chi2,
                "chi2_p_value": p_value,
                "dof": dof,
                "cramers_v": cramers_v,
            }
        )

    stats_df = (
        pd.DataFrame(stats_rows).set_index("feature")
        if stats_rows
        else pd.DataFrame()
    )

    return proportion_tables, stats_df

In [13]:
def compare_continuous_distributions(
    df,
    continuous_cols,
    split_col="split",
    reference_split="train",
    splits=("train", "val", "test", "no_split"),
):
    work = df[df[split_col].isin(splits)].copy()

    summary_rows = []
    for col in continuous_cols:
        if col not in work.columns:
            continue

        for s in splits:
            vals = pd.to_numeric(
                work.loc[work[split_col] == s, col],
                errors="coerce"
            )

            summary_rows.append(
                {
                    "feature": col,
                    "split": s,
                    "n": int(vals.notna().sum()),
                    "missing_rate": float(vals.isna().mean()),
                    "mean": float(vals.mean()),
                    "std": float(vals.std()),
                    "median": float(vals.median()),
                    "q25": float(vals.quantile(0.25)),
                    "q75": float(vals.quantile(0.75)),
                    "iqr": float(vals.quantile(0.75) - vals.quantile(0.25)),
                }
            )

    summary_df = pd.DataFrame(summary_rows)

    ks_rows = []
    for col in continuous_cols:
        if col not in work.columns:
            continue

        ref = pd.to_numeric(
            work.loc[work[split_col] == reference_split, col],
            errors="coerce"
        ).dropna()

        for s in splits:
            if s == reference_split:
                continue

            cur = pd.to_numeric(
                work.loc[work[split_col] == s, col],
                errors="coerce"
            ).dropna()

            if len(ref) == 0 or len(cur) == 0:
                ks_stat, p_value = np.nan, np.nan
            else:
                res = ks_2samp(ref, cur)
                ks_stat, p_value = float(res.statistic), float(res.pvalue)

            ks_rows.append(
                {
                    "feature": col,
                    "reference_split": reference_split,
                    "comparison_split": s,
                    "ks_stat": ks_stat,
                    "ks_p_value": p_value,
                }
            )

    ks_df = pd.DataFrame(ks_rows)
    return summary_df, ks_df

In [14]:
prop, stats_df = compare_categorical_distributions(df, categorical_features)
print(prop)
stats_df.sort_values("cramers_v", ascending=False)

{'location_setting': split                train       val      test  no_split
location_setting                                        
emergency         0.314743  0.364894  0.362183  0.285444
inpatient         0.481628  0.411371  0.404814  0.524145
outpatient        0.171411  0.185473  0.194598  0.163487
procedural        0.032218  0.038262  0.038405  0.026923, 'race_ethnicity': split              train       val      test  no_split
race_ethnicity                                        
asian           0.035902  0.028967  0.028115  0.030876
black           0.159489  0.157371  0.155458  0.190640
hispanic        0.314674  0.292045  0.303014  0.298276
other           0.072742  0.082144  0.083976  0.075614
unknown         0.123449  0.140078  0.141125  0.120009
white           0.293743  0.299395  0.288313  0.284585, 'sex': split     train       val      test  no_split
sex                                          
female  0.46256  0.509295  0.501838   0.44727
male    0.53744  0.490705  0.498

,n_levels,chi2,chi2_p_value,dof,cramers_v
feature,,,,,
location_setting,4,358.575554,9.259858e-72,9,0.034136
sex,2,89.719467,2.516580e-19,3,0.029448
race_ethnicity,6,170.616144,1.838464e-28,15,0.022776


In [15]:
summary_df, ks_df = compare_continuous_distributions(df, continuous_features)
ks_df.sort_values("ks_stat", ascending=False).reset_index(drop=True)

,feature,reference_split,comparison_split,ks_stat,ks_p_value
0,ventricular_rate,train,test,0.066773,4.564786e-20
1,qt_corrected,train,test,0.066191,1.001182e-19
2,atrial_rate,train,test,0.063400,5.463886e-18
3,ventricular_rate,train,no_split,0.063320,1.765841e-49
4,atrial_rate,train,no_split,0.061944,3.955587e-47
5,atrial_rate,train,val,0.061177,1.706428e-14
6,ventricular_rate,train,val,0.059532,7.749202e-14
7,age_at_ecg,train,val,0.052528,7.233336e-11
8,qt_corrected,train,val,0.049972,7.092891e-10
9,age_at_ecg,train,no_split,0.043466,1.558796e-23


In [16]:
summary_df

,feature,split,n,missing_rate,mean,std,median,q25,q75,iqr
0,ventricular_rate,train,72475,0.000000,83.379000,20.388859,81.0,69.0,95.0,26.0
1,ventricular_rate,val,4626,0.000000,80.963900,20.437194,78.0,66.0,92.0,26.0
2,ventricular_rate,test,5442,0.000000,80.810180,20.132099,78.0,66.0,92.0,26.0
3,ventricular_rate,no_split,17457,0.000000,85.928281,20.392441,84.0,71.0,98.0,27.0
4,atrial_rate,train,72023,0.006237,89.787151,44.007286,81.0,69.0,97.0,28.0
5,atrial_rate,val,4595,0.006701,86.525354,41.996579,78.0,66.0,93.0,27.0
6,atrial_rate,test,5397,0.008269,87.093571,43.831793,78.0,66.0,94.0,28.0
7,atrial_rate,no_split,17372,0.004869,92.243150,44.000479,85.0,71.0,99.0,28.0
8,pr_interval,train,64856,0.105126,159.665860,32.337435,154.0,138.0,174.0,36.0
9,pr_interval,val,4157,0.101383,160.420014,32.001228,156.0,140.0,176.0,36.0


Train, validation, and test are not identical, but the differences are small. For categorical variables (location_setting, sex, race_ethnicity), effect sizes are low, and for continuous variables (ventricular_rate, atrial_rate, qt_corrected), KS values are also low. In practice, the splits look consistent enough for training and evaluation.